# 02 — Build and verify initial Silver

**Live Silver verification: pending.** The user has reported successful live initial
Bronze verification (600 / 30,600 / 30,000 / 94,572 rows, zero corrupt rows and correct
source provenance). No live AIDP Silver execution has been performed by the author.

Run after notebook 01 reports BRONZE PASS, with project wheel **0.2.2** installed on AIDP compute. Notebook 00 is optional.
The reusable modules own all parsing, quality, version selection and Delta writes.
No notebook monkeypatches are needed. This notebook processes initial Bronze only.
Bronze v2 migration gate: run the separate migration prepare/apply/validation cell first. Do not run registration or Silver cells against the existing v1 registrations. See `docs/bronze-v2-migration.md`.


## One configuration section

Keep source configuration aligned with notebooks 00 and 01. The reference date
is fixed. Managed Silver uses AIDP catalog placement, not the source bucket.
The current demo uses a bounded driver-side quality engine; MAX_DRIVER_ROWS
prevents silently collecting an unexpectedly large batch. For larger datasets,
port the same contracts to distributed transforms before raising the guard.

In [ ]:
# CONFIGURATION: edit environment values only here; match notebooks 00 and 01.
# These are non-secret values verified in docs/object-storage-execution.md.
CONFIG = {
    "catalog": "lakehouse_demo",
    "namespace": "frgpslcqn7sm",  # Object Storage namespace, NOT oraemeaaidata2
    "bucket": "CristinaVaras-bucket-AIDP",
    "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
    "reference_date": "2026-10-28",  # Fixed synthetic-data reference date
    "seed": 42,
    "batch_id": "initial",         # Do not include incremental in this milestone
}
# Optional: absolute driver-visible checkout path when using source imports.
# Leave None when the project wheel is installed on the AIDP compute cluster.
REPO_ROOT = None

MAX_DRIVER_ROWS = 250_000  # Operational memory guard for the bounded demo, not a quality threshold.


In [ ]:
from pathlib import Path
import sys
import json
if REPO_ROOT is not None:
    sys.path.insert(0, str(Path(REPO_ROOT) / "src"))
from aidp_lakehouse import __version__
from aidp_lakehouse.aidp_bronze import BronzeConfig
from aidp_lakehouse.silver_spark import SilverConfig, build_initial_silver
if "spark" not in globals():
    raise RuntimeError("Attach authenticated AIDP Spark/Delta compute before running this notebook")
print("Installed project version:", __version__)
if __version__ not in ("0.2.2",):
    raise RuntimeError("Install the verified 0.2.2 wheel and restart the Python session")
cfg = SilverConfig(BronzeConfig(**CONFIG), max_driver_rows=MAX_DRIVER_ROWS)


## Transform and publish the initial snapshot

The helper re-verifies unchanged external Bronze definitions, counts and exact file
provenance, sets UTC, and parses the initial snapshot. It validates source ranges,
chronology, relationships, immutable acceptance values and version conflicts.

It preflights all five targets before writing: they must be managed Delta with the
exact contract, unique keys and either empty contents or an identical subset of the
planned snapshot. Unexpected initial rejects/duplicates stop before publication and
include an explicit quality preview. No blanket overwrite, reset, matched-row update,
or source mutation is used. Inserts use Delta MERGE with only WHEN NOT MATCHED.
The job must have a single writer; per-table commits are not a five-table transaction.

In [ ]:
first = build_initial_silver(spark, cfg)
print(json.dumps(first, indent=2, default=str))


## Prove initial-batch rerun idempotence

This intentionally rereads the same initial Bronze snapshot and runs the same
publication path again. It requires identical business fingerprints and summary
dispositions, and zero new target rows. It never updates or removes existing rows.

In [ ]:
second = build_initial_silver(spark, cfg)
assert first["business_fingerprints"] == second["business_fingerprints"]
for entity in ("customers", "orders", "deliveries", "events"):
    for metric in ("raw_rows", "duplicate_removals", "rejected_rows", "accepted_rows", "resulting_silver_rows"):
        assert first["summary"][entity][metric] == second["summary"][entity][metric], (entity, metric)
    assert second["summary"][entity]["inserted_rows"] == 0
assert second["quarantine"]["inserted_rows"] == 0
assert second["quarantine"]["resulting_silver_rows"] == 0
print("Initial rerun: identical business results, zero new target rows.")


## Inspect actual managed definitions, typed samples and quarantine

Expected clean initial Silver counts remain customers=600, orders=30,600,
deliveries=30,000, events=94,572; quarantine is empty. The JSON summary reconciles
each entity as raw = duplicate removals + rejected source rows + accepted rows.
Quarantine groups identical rejected source records with an occurrence count;
SUM(occurrence_count) is rejected source rows, not the number of rejection reasons.

In [ ]:
for entity in ("customers", "orders", "deliveries", "events", "quarantine"):
    name = f"{CONFIG['catalog']}.silver.{entity}"
    spark.sql(f"DESCRIBE TABLE EXTENDED {name}").show(100, truncate=False)
    spark.sql(f"SELECT * FROM {name} LIMIT 5").show(truncate=False)
print("Silver runtime verification completed in this attached AIDP session.")


## References and boundaries

See `docs/aidp-silver-setup.md` for explicit contracts, quality dispositions,
installation, local test evidence and future incremental design.

Managed DDL follows [Oracle AIDP SQL Grammar](https://docs.oracle.com/en/cloud/paas/ai-data-platform/aidug/sql-grammar.html).
Insert-only publication follows [Delta MERGE](https://docs.delta.io/delta-update/).
Silver execution remains pending until these cells run in your actual AIDP session.
No incremental files are read/uploaded and no Gold, ML, agents or Autonomous
publication is implemented.